AI Agent:

An AI agent is a software program that uses artificial intelligence to autonomously perceive its environment, reason, plan, and act to achieve specific goals, often with minimal human intervention

Tools

In AI agent systems, a tool is a predefined function or external service that an agent can call to perform specific actions or access information beyond its core capabilities. Tools bridge the gap between an agent's reasoning and real-world action, allowing it to retrieve data, execute code, interact with APIs, or perform other specialized tasks needed to fulfill a user's request.

PydanticAI is a Python framework for building LLM-powered agents with structured, type-safe inputs and outputs using Pydantic models.
PydanticAI = LLM + tools + strict data validation (via Pydantic)

In [1]:
import requests
import os
from dotenv import load_dotenv
load_dotenv()
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
API_KEY = os.getenv("OPENWEATHER_API_KEY")

def find_weather(city: str) -> dict:
    """ this function returns current weather forecast for a given city"""
    units = "metric"
    params = {
        'q' : city,
        'appid' : API_KEY,
        'units' : units
    }

    response = requests.get(BASE_URL, params = params)
    result = response.json()
    return result

In [2]:
output = find_weather("London")
print(output)

{'coord': {'lon': -0.1257, 'lat': 51.5085}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 11.67, 'feels_like': 11.03, 'temp_min': 10.34, 'temp_max': 12.83, 'pressure': 1013, 'humidity': 82, 'sea_level': 1013, 'grnd_level': 1008}, 'visibility': 10000, 'wind': {'speed': 2.57, 'deg': 30}, 'clouds': {'all': 75}, 'dt': 1777965038, 'sys': {'type': 2, 'id': 2075535, 'country': 'GB', 'sunrise': 1777955116, 'sunset': 1778009345}, 'timezone': 3600, 'id': 2643743, 'name': 'London', 'cod': 200}


In [3]:
print(output["name"])
print(output["weather"][0]["description"].capitalize())
print(output["main"]["temp"])

London
Broken clouds
11.67


In [4]:
import os
import requests
from pydantic import BaseModel # From Pydantic, used to define structured data with validation
from pydantic_ai import Agent, RunContext
# Agent: Core class to create an AI agent.
# RunContext: Provides runtime context (like metadata, memory, etc.) when tools are called.
from pydantic_ai.settings import ModelSettings # Used to configure model behavior (e.g., temperature).

# Without RunContext, your tool only knows: -> city: str
# But in real-world scenarios, tools often need more context:

# Who asked the question?
# What was the previous conversation?
# Any shared memory/state?
# Logging / tracing info?


In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [6]:
class WeatherForecast(BaseModel):
    location: str
    description: str
    temperature_celsius : float

# This defines the structured response format for your weather tool.
# Pydantic validates data automatically. 

In [7]:
weather_app = Agent(
    model="groq:llama-3.1-8b-instant", # Initializes the agent using the Groq-hosted LLaMA 3.1 model.
    model_settings=ModelSettings(temperature=0.2), # 0.2 → more deterministic, less creative.
    output_type=str, # Final response from agent will be a string (not structured).
    system_prompt=(
        "You are a helpful weather assistant. "
        "Use the 'get_weather_forecast' tool to fetch weather. "
        "Respond in a clean, friendly, slightly funny tone."
    ) # This is the instruction given to the LLM
)

In [ ]:
@weather_app.tool # Registers this function as a tool the agent can call.
def get_weather_forecast(ctx: RunContext, city: str) -> WeatherForecast:
    url = "https://api.openweathermap.org/data/2.5/weather"
    api_key = os.getenv("OPENWEATHER_API_KEY")

    params = {
        "q": city,
        "appid": api_key,
        "units": "metric"
    }

    response = requests.get(url, params=params)
    result = response.json()

    return WeatherForecast(
        location=result["name"],
        description=result["weather"][0]["description"].capitalize(),
        temperature_celsius=result["main"]["temp"]
    )


In [9]:
agent_response = await weather_app.run("What is the weather in Chennai?") # Sends user query to agent
# Agent:

# Reads prompt
# Decides to call tool
# Calls get_weather_forecast("Chennai")
# Gets structured data
# Converts to final response (string)

print(agent_response.output)

It looks like it's going to be a lovely day in Chennai! Scattered clouds and a temperature of 34.65 degrees Celsius. Perfect weather to grab a cup of filter coffee and enjoy the sun.


In [10]:
question = input("🌤️ Ask about the weather: ")
result = await weather_app.run(question)
print("\n📍 Forecast:", result.output)


📍 Forecast: It looks like it's a bit cloudy in Haridwar today. The temperature is a pleasant 22.44 degrees Celsius. Would you like to know more about the weather forecast for the next few days?


In [ ]:
"""
Weather Assistant using Pydantic-AI and OpenWeatherMap API
This script creates a conversational agent that can respond to weather-related queries
using the OpenWeatherMap API and a Groq-hosted LLaMA model.
"""

# Standard libraries
import os
import requests

# Pydantic model for structured data
from pydantic import BaseModel

# Core AI libraries from pydantic_ai
from pydantic_ai import Agent, RunContext
from pydantic_ai.settings import ModelSettings

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# 1. Define the output schema of the tool using Pydantic
class WeatherForecast(BaseModel):
    location: str
    description: str
    temperature_celsius: float

# 2. Create the AI agent using Groq’s LLaMA 3 model
weather_agent = Agent(
    model="groq:llama-3.1-8b-instant",
    model_settings=ModelSettings(temperature=0.2),
    output_type=WeatherForecast,
    system_prompt=(
    "You are a weather assistant.\n"
    "ALWAYS use the get_weather_forecast tool for weather queries.\n"
    "Do NOT return function calls.\n"
    "Return only final answers in clean text."
    )
)

# 3. Register a tool with the agent to fetch real-time weather using OpenWeatherMap API
@weather_agent.tool
def get_weather_forecast(ctx: RunContext, city: str) -> WeatherForecast:
    """
    Tool: get_weather_forecast
    Description: Fetches current weather for a city using the OpenWeatherMap API.
    """
    url = "https://api.openweathermap.org/data/2.5/weather"
    
    # Replace this with your own API key for production use
    api_key = os.getenv("OPENWEATHER_API_KEY")
    
    # Query parameters
    params = {
        'q': city,
        'appid': api_key,
        'units': 'metric'
    }

    # Send request to weather API
    res = requests.get(url, params=params).json()

    # Return the formatted weather information
    return WeatherForecast(
        location=res["name"],
        description=res["weather"][0]["description"].capitalize(),
        temperature_celsius=res["main"]["temp"]
    )

# 4. Run continuous user interaction loop
async def main():
    print("🌦️  Weather Assistant is ready! Type 'exit' to quit.")
    print("‒" * 50)

    while True:
        question = input("🌤️ Ask about the weather: ").strip()
        if question.lower() in {"exit", "quit", ""}:
            print("\n👋 Exiting weather assistant. Have a nice day!")
            break

        try:
            result = await weather_agent.run(question)
            forecast = result.output
            print(f"\n📍 Forecast : {forecast.location}: {forecast.description}, {forecast.temperature_celsius}°C")
        except Exception as e:
            print("⚠️ Error:", str(e))

        print("‒" * 50)

if __name__ == "__main__":
    await main()

🌦️  Weather Assistant is ready! Type 'exit' to quit.
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒



📍 Forecast : Haridwar: The current weather in Haridwar is mostly sunny with a high of 28 degrees Celsius., 28.0°C
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒

📍 Forecast : Allahabad: The current weather in Allahabad is partly cloudy with a high chance of rain., 25.0°C
‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒‒

👋 Exiting weather assistant. Have a nice day!
